# Clase 160 — Stable Diffusion XL + ControlNet

Fallback CPU-friendly: implementamos **forward diffusion** sobre imagen 32×32 sintética, un denoiser MLP, y simulamos ControlNet con Canny edges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)

DIFFUSERS_OK = False
try:
    import diffusers  # noqa
    DIFFUSERS_OK = True
    print('diffusers disponible (pero requiere GPU para SDXL real)')
except Exception:
    print('diffusers no disponible → usando fallback DDPM toy')

## 1. Imagen sintética 32×32

In [ ]:
# Imagen: círculo blanco sobre fondo gris
H = W = 32
yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
img0 = np.where(((yy - 16)**2 + (xx - 16)**2) < 64, 0.9, 0.2).astype(np.float32)
img0 = img0 + rng.normal(0, 0.02, img0.shape)
print('img shape:', img0.shape, '| range:', img0.min().round(2), img0.max().round(2))

## 2. Forward diffusion (DDPM toy)

$$ x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, I) $$

In [ ]:
T = 100
betas = np.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = np.cumprod(alphas)

def q_sample(x0, t):
    a_bar = alpha_bars[t]
    noise = rng.standard_normal(x0.shape)
    return np.sqrt(a_bar) * x0 + np.sqrt(1 - a_bar) * noise, noise

fig, axes = plt.subplots(1, 6, figsize=(12, 2))
for ax, t in zip(axes, [0, 20, 40, 60, 80, 99]):
    xt, _ = q_sample(img0, t)
    ax.imshow(xt, cmap='gray', vmin=-2, vmax=2); ax.set_title(f't={t}'); ax.axis('off')
plt.suptitle('Forward diffusion: x0 → ruido puro'); plt.tight_layout(); plt.show()

## 3. Denoiser U-Net simulado (sklearn MLP)

Entrenamos un MLP a predecir el ruido `ε` dado `(x_t, t)`. Una sola imagen (overfit a propósito).

In [ ]:
from sklearn.neural_network import MLPRegressor

# Dataset: muchos (x_t, t, eps) de la misma imagen base
N = 500
Xs, Ys = [], []
for _ in range(N):
    t = rng.integers(0, T)
    xt, eps = q_sample(img0, t)
    feat = np.concatenate([xt.flatten(), [t / T]])
    Xs.append(feat); Ys.append(eps.flatten())
Xs, Ys = np.array(Xs), np.array(Ys)
print(f'train shape: X={Xs.shape}, Y={Ys.shape}')

mlp = MLPRegressor(hidden_layer_sizes=(256, 256), max_iter=80, random_state=42, verbose=False)
mlp.fit(Xs, Ys)
print(f'train R²: {mlp.score(Xs, Ys):.3f}')

## 4. Reverse sampling (DDPM step)

$$ x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar\alpha_t}}\,\hat\epsilon_\theta(x_t, t)\right) + \sigma_t z $$

In [ ]:
x = rng.standard_normal(img0.shape)   # ruido puro
snapshots = []
for t in reversed(range(T)):
    feat = np.concatenate([x.flatten(), [t / T]]).reshape(1, -1)
    eps_hat = mlp.predict(feat).reshape(H, W)
    a, ab, b = alphas[t], alpha_bars[t], betas[t]
    mean = (1 / np.sqrt(a)) * (x - (b / np.sqrt(1 - ab)) * eps_hat)
    z = rng.standard_normal(x.shape) if t > 0 else 0
    x = mean + np.sqrt(b) * z
    if t in [99, 80, 60, 40, 20, 0]: snapshots.append((t, x.copy()))

fig, axes = plt.subplots(1, 6, figsize=(12, 2))
for ax, (t, xt) in zip(axes, snapshots[::-1]):
    ax.imshow(xt, cmap='gray'); ax.set_title(f't={t}'); ax.axis('off')
plt.suptitle('Reverse: ruido → imagen aprendida'); plt.tight_layout(); plt.show()

## 5. ControlNet conceptual: edge map como condicionamiento

ControlNet inyecta señales extras (Canny, depth, pose, scribble) que guían la generación. Mostramos el preprocesado de Canny edges.

In [ ]:
from scipy import ndimage

def canny_lite(img, low=0.1, high=0.3):
    """Mini Canny: smooth + sobel + threshold."""
    sm = ndimage.gaussian_filter(img, 1.0)
    gx = ndimage.sobel(sm, axis=1); gy = ndimage.sobel(sm, axis=0)
    mag = np.hypot(gx, gy); mag = mag / mag.max()
    return (mag > low).astype(float)

edges = canny_lite(img0)
fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(img0, cmap='gray'); ax[0].set_title('input'); ax[0].axis('off')
ax[1].imshow(edges, cmap='gray'); ax[1].set_title('Canny edges (control)'); ax[1].axis('off')
plt.show()
print('En ControlNet real: edges se inyecta al U-Net via zero-conv blocks.')

## 6. API real SDXL + ControlNet (requiere GPU)

```python
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
controlnet = ControlNetModel.from_pretrained('diffusers/controlnet-canny-sdxl-1.0')
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0', controlnet=controlnet,
    torch_dtype=torch.float16).to('cuda')
image = pipe('a futuristic cat, cinematic', image=canny_image,
             num_inference_steps=30, guidance_scale=7.5).images[0]
```

## Ejercicio guiado

1. Variar T ∈ {50, 100, 500} y comparar calidad del sampling.
2. Implementar DDIM (sampling deterministico) en vez de DDPM.
3. Probar con shape distinto (cuadrado, triángulo).

## Conclusiones

- Diffusion = forward Markov (add noise) + reverse learned (denoise).
- SDXL = U-Net + latent space (VAE) + 2 text encoders + refiner opcional.
- ControlNet desbloquea control espacial (composición, pose, profundidad).